# 04 — Modelo Preditivo de Perfil Financeiro

**Projeto:** Financial Behavior Intelligence  
**Objetivo:** Treinar um classificador que prediz o perfil financeiro de um usuário (`saver`, `debtor`, `balanced`) com base em seu comportamento histórico de transações.

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection   import train_test_split, cross_val_score
from sklearn.ensemble          import RandomForestClassifier
from sklearn.preprocessing     import LabelEncoder
from sklearn.metrics           import (
    classification_report, confusion_matrix,
    accuracy_score, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42

## 1. Carregar Features e Target

> **Atenção:** O target `User_Profile` é gerado a partir das regras do `data_generator.py`.  
> Ele deve ser mergeado aqui com base no `User_ID`.

In [ ]:
features = pd.read_csv('../data/processed/user_features.csv')

# ADD: carregar o mapeamento User_ID -> User_Profile gerado pelo data_generator
# Exemplo:
# profiles = pd.read_csv('../data/processed/user_profiles.csv')  # User_ID, User_Profile
# features = features.merge(profiles, on='User_ID', how='left')

print('Features shape:', features.shape)
features.head()

In [ ]:
# Remover usuários sem perfil definido (unknown)
df_model = features[features['User_Profile'].notna()].copy()
df_model = df_model[df_model['User_Profile'] != 'unknown']

print('Usuários para treino/teste:', len(df_model))
print('Distribuição do target:')
print(df_model['User_Profile'].value_counts())

## 2. Preparar X e y

In [ ]:
FEATURE_COLS = [
    'avg_monthly_debit',
    'avg_monthly_credit',
    'avg_saldo',
    'std_saldo',
    'spending_volatility',
    'pct_meses_negativo',
    'savings_rate',
    'top_category_pct',
    'num_fixed_expenses',
    'has_investment',
    'credit_card_ratio'
]

X = df_model[FEATURE_COLS].fillna(0)
y = df_model['User_Profile']

le = LabelEncoder()
y_enc = le.fit_transform(y)

print('Classes:', le.classes_)
print('X shape:', X.shape)

## 3. Split Treino / Teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=RANDOM_STATE, stratify=y_enc
)

print(f'Treino: {len(X_train)} | Teste: {len(X_test)}')

## 4. Treinar Random Forest

In [ ]:
clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=6,
    random_state=RANDOM_STATE
)

clf.fit(X_train, y_train)
print('Modelo treinado.')

## 5. Avaliação

In [ ]:
y_pred = clf.predict(X_test)

print(f'Acurácia no teste: {accuracy_score(y_test, y_pred):.2%}')
print()
print('Relatório por classe:')
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Validação cruzada (mais robusto com poucas amostras)
cv_scores = cross_val_score(clf, X, y_enc, cv=5, scoring='accuracy')
print(f'Cross-val accuracy: {cv_scores.mean():.2%} (+/- {cv_scores.std():.2%})')

In [ ]:
# Matriz de Confusão
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=le.classes_,
    cmap='Blues', ax=ax
)
ax.set_title('Matriz de Confusão — Perfil Financeiro')
plt.tight_layout()
plt.show()

## 6. Feature Importance (Interpretabilidade)

In [ ]:
importance = pd.Series(clf.feature_importances_, index=FEATURE_COLS)
importance = importance.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importance.plot(kind='barh', ax=ax, color='teal')
ax.set_title('Importância das Features — Random Forest')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

print('\nTop 3 features mais relevantes:')
print(importance.sort_values(ascending=False).head(3))

## 7. Salvar Predições

In [ ]:
all_pred = clf.predict(X)
df_model = df_model.copy()
df_model['perfil_previsto'] = le.inverse_transform(all_pred)

output = df_model[['User_ID', 'User_Profile', 'perfil_previsto'] + FEATURE_COLS]
output.to_csv('../data/processed/user_predictions.csv', index=False)
print('Predições salvas em data/processed/user_predictions.csv')